In [12]:
import pandas as pd
import numpy as np
from google.colab import files

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

import statsmodels.api as sm

# Upload clean dataset
uploaded = files.upload()
df = pd.read_csv("ready_df.csv")
df["gameDate"] = pd.to_datetime(df["gameDate"])
df = df.sort_values("gameDate").reset_index(drop=True)



Saving ready_df.csv to ready_df (4).csv


In [13]:
df["diff_eFG"] = df["home_eFGpct"] - df["away_eFGpct"]
df["diff_TO"]  = df["home_turnoverPercentage"] - df["away_turnoverPercentage"]
df["diff_ORB"] = df["home_OBRpct"] - df["away_OBRpct"]
df["diff_DRB"] = df["home_DBRpct"] - df["away_DBRpct"]
df["diff_win"] = df["home_winPct"] - df["away_winPct"]

df["diff_FGpct"] = df["home_fieldGoalsPercentage"] - df["away_fieldGoalsPercentage"]
df["diff_3Ppct"] = df["home_threePointersPercentage"] - df["away_threePointersPercentage"]
df["diff_FTpct"] = df["home_freeThrowsPercentage"] - df["away_freeThrowsPercentage"]
df["diff_TOV"]   = df["home_turnovers"] - df["away_turnovers"]
df["diff_OR"]    = df["home_reboundsOffensive"] - df["away_reboundsOffensive"]
df["diff_DR"]    = df["home_reboundsDefensive"] - df["away_reboundsDefensive"]

x_cols = [
    "diff_eFG","diff_TO","diff_ORB","diff_DRB","diff_win",
    "diff_FGpct","diff_3Ppct","diff_FTpct",
    "diff_TOV","diff_OR","diff_DR",
    "home_winPct","away_winPct"
]
y_cols = ["spread", "total", "OREB"]

dat = df[["gameDate"] + y_cols + x_cols].dropna().copy()
dat["gameDate"] = pd.to_datetime(dat["gameDate"])
dat = dat.sort_values("gameDate").reset_index(drop=True)

# 3) Time-based split
# =========================
cut_date = dat["gameDate"].quantile(0.80)

train = dat[dat["gameDate"] <= cut_date].copy()
test  = dat[dat["gameDate"] >  cut_date].copy()

X_train_raw = train[x_cols]
X_test_raw  = test[x_cols]

print("Train cutoff:", cut_date)
print("Train rows:", len(train), "| Test rows:", len(test))
print("Train dates:", train["gameDate"].min(), "to", train["gameDate"].max())
print("Test dates :", test["gameDate"].min(),  "to", test["gameDate"].max())

# 4) CV on TRAIN only
# =========================
tscv = TimeSeriesSplit(n_splits=5)

# 5) Preprocessing pipelines
# =========================
preprocess_linear_regular = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), x_cols)
    ],
    remainder="drop"
)

preprocess_linear_poly = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler())
        ]), x_cols)
    ],
    remainder="drop"
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), x_cols)
    ],
    remainder="drop"
)
# 6) OLS diagnostics (R2/AdjR2) on ORIGINAL predictors
# =========================
def adj_r2(r2, n, p):
    if n <= p + 1:
        return np.nan
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

def fit_ols_statsmodels_originalX(outcome):
    y_train = train[outcome].astype(float).values
    y_test  = test[outcome].astype(float).values

    Xtr = X_train_raw.copy().fillna(X_train_raw.median(numeric_only=True))
    Xte = X_test_raw.copy().fillna(X_train_raw.median(numeric_only=True))

    Xtr_sm = sm.add_constant(Xtr)
    Xte_sm = sm.add_constant(Xte, has_constant="add")

    model = sm.OLS(y_train, Xtr_sm).fit()
    pred_test = model.predict(Xte_sm)

    mae_test = mean_absolute_error(y_test, pred_test)

    r2_train = model.rsquared
    adjr2_train = model.rsquared_adj

    ss_res = np.sum((y_test - pred_test) ** 2)
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
    r2_test = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    adjr2_test = adj_r2(r2_test, n=len(y_test), p=len(x_cols))

    return {
        "mae_test": mae_test,
        "r2_train": r2_train,
        "adjr2_train": adjr2_train,
        "r2_test": r2_test,
        "adjr2_test": adjr2_test
    }
# 7) Fit + compare models for one outcome
# =========================
def fit_and_compare(outcome: str):
    y_train = train[outcome].values
    y_test  = test[outcome].values

    rows = []
    best_params = {}

    # OLS regular
    ols_reg = Pipeline(steps=[("prep", preprocess_linear_regular),
                             ("model", LinearRegression())])
    ols_reg.fit(X_train_raw, y_train)
    pred = ols_reg.predict(X_test_raw)
    rows.append(("OLS (regular)", mean_absolute_error(y_test, pred)))

    # OLS poly
    ols_poly = Pipeline(steps=[("prep", preprocess_linear_poly),
                              ("model", LinearRegression())])
    ols_poly.fit(X_train_raw, y_train)
    pred = ols_poly.predict(X_test_raw)
    rows.append(("OLS (poly+interactions)", mean_absolute_error(y_test, pred)))

    # Ridge regular (tuned alpha)
    ridge_reg = Pipeline(steps=[("prep", preprocess_linear_regular),
                               ("model", Ridge(random_state=1))])
    ridge_param = {"model__alpha": np.logspace(-3, 3, 30)}
    ridge_search_reg = RandomizedSearchCV(
        ridge_reg, ridge_param, n_iter=18,
        scoring="neg_mean_absolute_error", cv=tscv, random_state=1, n_jobs=-1
    )
    ridge_search_reg.fit(X_train_raw, y_train)
    pred = ridge_search_reg.best_estimator_.predict(X_test_raw)
    rows.append(("Ridge (regular, tuned)", mean_absolute_error(y_test, pred)))
    best_params["ridge_regular"] = ridge_search_reg.best_params_

    # Ridge poly (tuned alpha)
    ridge_poly = Pipeline(steps=[("prep", preprocess_linear_poly),
                                ("model", Ridge(random_state=1))])
    ridge_search_poly = RandomizedSearchCV(
        ridge_poly, ridge_param, n_iter=18,
        scoring="neg_mean_absolute_error", cv=tscv, random_state=1, n_jobs=-1
    )
    ridge_search_poly.fit(X_train_raw, y_train)
    pred = ridge_search_poly.best_estimator_.predict(X_test_raw)
    rows.append(("Ridge (poly, tuned)", mean_absolute_error(y_test, pred)))
    best_params["ridge_poly"] = ridge_search_poly.best_params_

    # Lasso regular (tuned alpha)
    lasso_reg = Pipeline(steps=[("prep", preprocess_linear_regular),
                               ("model", Lasso(max_iter=30000, random_state=1))])
    lasso_param = {"model__alpha": np.logspace(-4, 1, 40)}
    lasso_search_reg = RandomizedSearchCV(
        lasso_reg, lasso_param, n_iter=18,
        scoring="neg_mean_absolute_error", cv=tscv, random_state=1, n_jobs=-1
    )
    lasso_search_reg.fit(X_train_raw, y_train)
    pred = lasso_search_reg.best_estimator_.predict(X_test_raw)
    rows.append(("Lasso (regular, tuned)", mean_absolute_error(y_test, pred)))
    best_params["lasso_regular"] = lasso_search_reg.best_params_

    # Lasso poly (tuned alpha)
    lasso_poly = Pipeline(steps=[("prep", preprocess_linear_poly),
                                ("model", Lasso(max_iter=30000, random_state=1))])
    lasso_search_poly = RandomizedSearchCV(
        lasso_poly, lasso_param, n_iter=18,
        scoring="neg_mean_absolute_error", cv=tscv, random_state=1, n_jobs=-1
    )
    lasso_search_poly.fit(X_train_raw, y_train)
    pred = lasso_search_poly.best_estimator_.predict(X_test_raw)
    rows.append(("Lasso (poly, tuned)", mean_absolute_error(y_test, pred)))
    best_params["lasso_poly"] = lasso_search_poly.best_params_

    # Regression Tree (CART) tuned
    cart_pipe = Pipeline(steps=[("prep", preprocess_tree),
                               ("model", DecisionTreeRegressor(random_state=1))])
    cart_param = {
        "model__max_depth": [None, 3, 5, 8, 12, 20],
        "model__min_samples_leaf": [1, 2, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10, 20]
    }
    cart_search = RandomizedSearchCV(
        cart_pipe, cart_param, n_iter=25,
        scoring="neg_mean_absolute_error", cv=tscv, random_state=1, n_jobs=-1
    )
    cart_search.fit(X_train_raw, y_train)
    pred = cart_search.best_estimator_.predict(X_test_raw)
    rows.append(("Regression Tree (CART) (tuned)", mean_absolute_error(y_test, pred)))
    best_params["cart"] = cart_search.best_params_

    # Random Forest tuned
    rf_pipe = Pipeline(steps=[("prep", preprocess_tree),
                             ("model", RandomForestRegressor(
                                 n_estimators=800, random_state=1, n_jobs=-1
                             ))])
    rf_param = {
        "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.8],
        "model__min_samples_leaf": [1, 2, 5, 10, 20],
        "model__max_depth": [None, 5, 10, 20],
    }
    rf_search = RandomizedSearchCV(
        rf_pipe, rf_param, n_iter=18,
        scoring="neg_mean_absolute_error", cv=tscv, random_state=1, n_jobs=-1
    )
    rf_search.fit(X_train_raw, y_train)
    pred = rf_search.best_estimator_.predict(X_test_raw)
    rows.append(("Random Forest (tuned)", mean_absolute_error(y_test, pred)))
    best_params["rf"] = rf_search.best_params_

    # XGBoost tuned
    xgb_pipe = Pipeline(steps=[("prep", preprocess_tree),
                              ("model", XGBRegressor(
                                  objective="reg:squarederror",
                                  random_state=1, n_jobs=-1,
                                  subsample=0.8, colsample_bytree=0.8
                              ))])
    xgb_param = {
        "model__n_estimators": [300, 600, 900, 1200],
        "model__max_depth": [2, 3, 4, 6],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__min_child_weight": [1, 3, 5, 10],
        "model__reg_alpha": [0.0, 0.1, 0.5],
        "model__reg_lambda": [0.5, 1.0, 2.0, 5.0],
        "model__gamma": [0.0, 0.05, 0.1, 0.3],
    }
    xgb_search = RandomizedSearchCV(
        xgb_pipe, xgb_param, n_iter=25,
        scoring="neg_mean_absolute_error", cv=tscv, random_state=1, n_jobs=-1
    )
    xgb_search.fit(X_train_raw, y_train)
    pred = xgb_search.best_estimator_.predict(X_test_raw)
    rows.append(("XGBoost (tuned)", mean_absolute_error(y_test, pred)))
    best_params["xgb"] = xgb_search.best_params_

    leaderboard = pd.DataFrame(rows, columns=["model", "test_MAE"])
    leaderboard.insert(0, "outcome", outcome)
    leaderboard = leaderboard.sort_values("test_MAE").reset_index(drop=True)

    ols_diag = fit_ols_statsmodels_originalX(outcome)
    return leaderboard, best_params, ols_diag

# 8) Run all 3 outcomes
# =========================
all_tables = []
for outcome in ["spread", "total", "OREB"]:
    lb, best_params, ols_diag = fit_and_compare(outcome)
    all_tables.append(lb)

    print("\n==============================")
    print("Outcome:", outcome)
    print(lb)

    print("\nBest tuned params (CV on TRAIN):")
    for k, v in best_params.items():
        print(f"  {k}: {v}")

    print("\nOLS diagnostics using ORIGINAL predictors (TRAIN):")
    print("  R^2:", round(ols_diag["r2_train"], 4))
    print("  Adj R^2:", round(ols_diag["adjr2_train"], 4))

    print("\nOLS diagnostics using ORIGINAL predictors (TEST):")
    print("  R^2:", round(ols_diag["r2_test"], 4))
    print("  Adj R^2:", round(ols_diag["adjr2_test"], 4))

final_leaderboard = pd.concat(all_tables, ignore_index=True)
print("\n=== Combined leaderboard ===")
print(final_leaderboard.sort_values(["outcome", "test_MAE"]))

Train cutoff: 2025-12-23 00:00:00
Train rows: 1798 | Test rows: 447
Train dates: 2024-10-08 00:00:00 to 2025-12-23 00:00:00
Test dates : 2025-12-25 00:00:00 to 2026-02-27 00:00:00

Outcome: spread
  outcome                           model  test_MAE
0  spread         OLS (poly+interactions)  2.500728
1  spread             Ridge (poly, tuned)  2.501791
2  spread                   OLS (regular)  2.502889
3  spread          Ridge (regular, tuned)  2.502951
4  spread          Lasso (regular, tuned)  2.503263
5  spread             Lasso (poly, tuned)  2.539307
6  spread                 XGBoost (tuned)  3.033386
7  spread           Random Forest (tuned)  3.728097
8  spread  Regression Tree (CART) (tuned)  5.056049

Best tuned params (CV on TRAIN):
  ridge_regular: {'model__alpha': np.float64(0.002592943797404667)}
  ridge_poly: {'model__alpha': np.float64(0.11721022975334805)}
  lasso_regular: {'model__alpha': np.float64(0.00018047217668271703)}
  lasso_poly: {'model__alpha': np.float64(0.015